# Módulo 3 · Agente inteligente — Alertas y recomendación logística

Loop **ReAct determinista** (Observar → Razonar → Actuar) sin LLM externo: 100% reproducible y sin API keys. Las 4 tools integran los Módulos 1 y 2.

| Tool | Integra |
|---|---|
| `get_stock_status` | inventario simulado |
| `get_demand_forecast` | modelo del Módulo 2 |
| `run_optimization` | solver del Módulo 1 |
| `send_alert` | notificación simulada |

In [1]:
import sys; sys.path.append("..")
from src.agent import Tools, LogisticsAgent
from src.utils import set_seeds
set_seeds()
tools = Tools()
agent = LogisticsAgent(tools=tools)
out = agent.run()
print("Puntos en riesgo:", out["at_risk"])
print("Alertas enviadas:", len(out["alerts"]))
print("Criterio de parada:", out["stop_reason"])

Puntos en riesgo: ['D14', 'D6', 'D4', 'D17', 'D7', 'D3', 'D8', 'D2', 'D1']
Alertas enviadas: 9
Criterio de parada: all at-risk points attended


## Trazabilidad: el razonamiento paso a paso

Cada entrada del log registra el paso, la fase (`observe` / `reason` / `act` / `stop`) y su contenido. Permite reconstruir **por qué** el agente tomó cada decisión — auditable de principio a fin. Nótese que por cada punto en riesgo se llaman las tools `get_demand_forecast` y `get_stock_status` antes de decidir la severidad y enviar la alerta.

In [2]:
# Trazabilidad: log completo paso a paso
for t in out["trace"]:
    print(f"[{t['step']:>2}] {t['phase']:<8} {str(t['content'])[:130]}")

[ 0] observe  {'n_points': 20, 'n_at_risk': 9, 'at_risk': {'D14': 0.997, 'D6': 0.995, 'D4': 0.993, 'D17': 0.979, 'D7': 0.958, 'D3': 0.93, 'D8': 
[ 0] reason   9 points exceed risk threshold 0.5. Plan: for each, check demand forecast and nearest warehouse stock, then alert with severity fr
[ 1] act      {'tool': 'get_demand_forecast', 'args': {'point_id': 'D14'}, 'result': {'point_id': 'D14', 'weeks': 4, 'forecast': [300.8, 295.0, 
[ 1] act      {'tool': 'get_stock_status', 'args': {'warehouse_id': 'W5'}, 'result': {'warehouse_id': 'W5', 'stock': 797, 'capacity': 1314}}
[ 1] reason   D14: risk=1.00, stock=177.5, lead_time=10.0d, 4-week forecast=[300.8, 295.0, 287.0, 275.3] (stable/falling). Nearest warehouse W5 
[ 1] act      {'tool': 'send_alert', 'args': {'point_id': 'D14', 'severity': 'HIGH'}, 'result': {'delivered': True}}
[ 2] act      {'tool': 'get_demand_forecast', 'args': {'point_id': 'D6'}, 'result': {'point_id': 'D6', 'weeks': 4, 'forecast': [262.4, 254.5, 25
[ 2] act      {'t

**Lectura de la traza:** el agente detecta 9 puntos en riesgo, los atiende uno a uno con severidad sensible al stock del almacén más cercano (p. ej. recomienda jalar de un almacén secundario cuando el más cercano no cubre la demanda), y al detectar casos de severidad alta ejecuta una **re-optimización bajo +20 %** (paso 10) para validar factibilidad antes de parar.

In [3]:
# Ejemplo de alerta generada
out["alerts"][0]

{'point_id': 'D14',
 'message': 'Stockout risk 100%. Current stock 177.5 vs projected demand 298.9 (lead time 10.0d). Forecast next 4 weeks: [300.8, 295.0, 287.0, 275.3]. Nearest warehouse W5: 797/1314 units. Recommendation: replenish from W5 and pull from a secondary warehouse (W5 alone cannot cover the projected need).',
 'severity': 'HIGH',
 'ts': '2026-01-01T00:00:00+00:00'}

**Extensibilidad a LLM:** la política de razonamiento está aislada en `LogisticsAgent.run()`. Para usar un LLM bastaría reemplazar la selección de acción por una llamada con tool-calling (las tools ya tienen firma y registro estilo function-calling).